# Задача 4: Ансамбли методов ML
Объединение моделей для повышения Macro F1. 
В этом ноутбуке мы загрузим данные, проверим колонки и обучим ансамбли.

## Шаг 1. Импорт библиотек и загрузка данных
Мы используем `read_parquet` вместо `read_csv`.

In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, classification_report
import lightgbm as lgb

# Добавляем путь к папке src, чтобы импортировать функцию предобработки
sys.path.append(os.path.abspath('../src'))
# ВАЖНО: проверьте название функции в src/preprocess.py и исправьте, если она называется иначе
from preprocessing import clean_text 

# Загружаем данные из Parquet
train_df = pd.read_parquet('../data/train.parquet')
test_df = pd.read_parquet('../data/test.parquet')

# ВЫВОДИМ КОЛОНКИ, ЧТОБЫ ВЫ МОГЛИ ИХ ПРОВЕРИТЬ
print("=" * 50)
print("КОЛОНКИ В TRAIN:", train_df.columns.tolist())
print("КОЛОНКИ В TEST:", test_df.columns.tolist())
print("=" * 50)
print("Пример данных (первые 2 строки):")
print(train_df.head(2))
print("=" * 50)

In [ ]:
TEXT_COLUMN = 'CONTENT'  
LABEL_COLUMN = 'class' 

train_texts = [clean_text(str(text)) for text in train_df[TEXT_COLUMN]]
test_texts = [clean_text(str(text)) for text in test_df[TEXT_COLUMN]]

train_labels = train_df[LABEL_COLUMN].values
test_labels = test_df[LABEL_COLUMN].values

print(f"Данные загружены. Обучающая выборка: {len(train_texts)}, Тестовая: {len(test_texts)}")

## Шаг 2. Стекинг (Stacking)
Объединяем SVM, Логистическую регрессию и Наивный Байес. 
В качестве мета-модели используем Логистическую регрессию.

In [ ]:
stacking_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1, 2))),
    ('stacking', StackingClassifier(
        estimators=[
            ('svm', LinearSVC(random_state=42, dual=True)),
            ('lr', LogisticRegression(random_state=42, max_iter=1000)),
            ('nb', MultinomialNB())
        ],
        final_estimator=LogisticRegression(),
        cv=5,
        n_jobs=-1
    ))
])

print("Обучение Stacking...")
stacking_pipeline.fit(train_texts, train_labels)
y_pred_stack = stacking_pipeline.predict(test_texts)
macro_f1_stack = f1_score(test_labels, y_pred_stack, average='macro')

print(f"\nMacro F1 (Stacking): {macro_f1_stack:.4f}")
print(classification_report(test_labels, y_pred_stack))

## Шаг 3. Градиентный бустинг (LightGBM)
Альтернативный мощный алгоритм для текстовых данных.

In [ ]:
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(train_texts)
X_test_tfidf = vectorizer.transform(test_texts)

lgbm_model = lgb.LGBMClassifier(
    n_estimators=300, 
    learning_rate=0.1, 
    max_depth=8, 
    random_state=42, 
    verbose=-1
)

print("Обучение LightGBM...")
lgbm_model.fit(X_train_tfidf, train_labels)
y_pred_lgbm = lgbm_model.predict(X_test_tfidf)
macro_f1_lgbm = f1_score(test_labels, y_pred_lgbm, average='macro')

print(f"\nMacro F1 (LightGBM): {macro_f1_lgbm:.4f}")
print(classification_report(test_labels, y_pred_lgbm))

## Шаг 4. Сохранение результатов
Сохраняем метрики в папку `results/`, чтобы они попали в итоговый отчет.

In [ ]:
results_df = pd.DataFrame({
    'Model': ['Stacking (SVM+LR+NB)', 'LightGBM'],
    'Macro_F1': [macro_f1_stack, macro_f1_lgbm]
})

os.makedirs('../results', exist_ok=True)
results_df.to_csv('../results/ensemble_results.csv', index=False)

print("\nРезультаты сохранены в ../results/ensemble_results.csv")
print(results_df)